1. Helper function : Upsampling

In [ ]:
import numpy as np
from scipy.interpolate import RectBivariateSpline
from pyuvdata import UVBeam

def upsample_uvbeam_azza(
    beam: UVBeam,
    upsample_factor_za: int = 2,
    upsample_factor_az: int = 2,
    kx: int = 3,
    ky: int = 3,
) -> UVBeam:
    """
    Upsample a UVBeam defined on an (az, za) pixel grid to a finer grid.

    Parameters
    ----------
    beam : UVBeam
        Input beam with pixel_coordinate_system == 'az_za'.
        Assumes data_array shape: (Naxes_vec, Nfeeds, Nfreqs, Nza, Naz).
    upsample_factor_za : int
        Factor by which to increase the number of ZA samples.
    upsample_factor_az : int
        Factor by which to increase the number of AZ samples.
    kx, ky : int
        Spline degrees in ZA and AZ for RectBivariateSpline (1–5, typically 1–3).

    Returns
    -------
    beam_fine : UVBeam
        New beam with finer az/za grids and interpolated data_array.
    """
    if beam.pixel_coordinate_system != "az_za":
        raise ValueError("upsample_uvbeam_azza expects pixel_coordinate_system == 'az_za'")

    # Original grids
    az_old = beam.axis1_array  # (Naz,) azimuth (radians)
    za_old = beam.axis2_array  # (Nza,) zenith angle (radians)

    Naz_old = az_old.size
    Nza_old = za_old.size

    # Sanity: RectBivariateSpline requires strictly increasing axes
    if not (np.all(np.diff(za_old) > 0) and np.all(np.diff(az_old) > 0)):
        raise ValueError("za_old and az_old must be strictly increasing for RectBivariateSpline.")

    # New finer grids: same min/max, more samples
    Naz_new = Naz_old * upsample_factor_az
    Nza_new = Nza_old * upsample_factor_za

    az_new = np.linspace(az_old[0], az_old[-1], Naz_new)
    za_new = np.linspace(za_old[0], za_old[-1], Nza_new)

    # Prepare new beam and data array
    beam_fine = beam.copy()
    new_data = np.empty(
        (beam.Naxes_vec, beam.Nfeeds, beam.Nfreqs, Nza_new, Naz_new),
        dtype=beam.data_array.dtype,
    )

    # Loop over (vec, feed, freq) and spline-interpolate each 2D slice
    for ivec in range(beam.Naxes_vec):
        for ifeed in range(beam.Nfeeds):
            for ifreq in range(beam.Nfreqs):
                # slice_2d: (Nza_old, Naz_old)
                slice_2d = beam.data_array[ivec, ifeed, ifreq, :, :]

                # Real and imaginary parts separately
                spline_real = RectBivariateSpline(
                    za_old, az_old, slice_2d.real,
                    kx=kx, ky=ky
                )
                spline_imag = RectBivariateSpline(
                    za_old, az_old, slice_2d.imag,
                    kx=kx, ky=ky
                )

                # Evaluate on the finer grid: returns (Nza_new, Naz_new)
                vals_real = spline_real(za_new, az_new)
                vals_imag = spline_imag(za_new, az_new)

                new_slice = vals_real + 1j * vals_imag
                new_data[ivec, ifeed, ifreq, :, :] = new_slice.astype(beam.data_array.dtype)

    # Update beam metadata
    beam_fine.axis1_array = az_new
    beam_fine.axis2_array = za_new
    beam_fine.Naxes1 = Naz_new
    beam_fine.Naxes2 = Nza_new

    # For pixelized beams this is often used; adjust if your version uses a different attribute
    if hasattr(beam_fine, "Npixels"):
        beam_fine.Npixels = Naz_new * Nza_new

    beam_fine.data_array = new_data

    beam_fine.history += (
        f"\nUpsampled az/za grid by factors "
        f"(za x az) = ({upsample_factor_za} x {upsample_factor_az}) "
        f"using RectBivariateSpline(kx={kx}, ky={ky})."
    )

    return beam_fine


2. Helper function : resampling in azimuth, zenith angles

In [ ]:
import numpy as np
from scipy.interpolate import RectBivariateSpline
from pyuvdata import UVBeam

def resample_uvbeam_azza(
    beam: UVBeam,
    za_new: np.ndarray,
    az_new: np.ndarray,
    kx: int = 3,
    ky: int = 3,
) -> UVBeam:
    """
    Resample a UVBeam defined on an (az, za) pixel grid
    onto a new (za_new, az_new) grid using 2D splines.

    Parameters
    ----------
    beam : UVBeam
        Input beam with pixel_coordinate_system == 'az_za'.
        Assumes data_array shape: (Naxes_vec, Nfeeds, Nfreqs, Nza_old, Naz_old).
    za_new : array_like
        New zenith-angle grid (radians), 1D, strictly increasing.
    az_new : array_like
        New azimuth grid (radians), 1D, strictly increasing.
    kx, ky : int
        Spline degrees in ZA and AZ for RectBivariateSpline (1–5, typically 1–3).

    Returns
    -------
    beam_resampled : UVBeam
        New beam with (axis2_array, axis1_array) replaced by (za_new, az_new),
        and data_array interpolated on that grid.
    """
    if beam.pixel_coordinate_system != "az_za":
        raise ValueError("resample_uvbeam_azza expects pixel_coordinate_system == 'az_za'")
    
    az_old = beam.axis1_array  # (Naz_old,) azimuth (radians)
    za_old = beam.axis2_array  # (Nza_old,) zenith angle (radians)
    
    print("za_new, az_new", np.rad2deg(za_new), np.rad2deg(az_new))
    print("za_old, az_old", np.rad2deg(za_old), np.rad2deg(az_old))

    Naz_old = az_old.size
    Nza_old = za_old.size

    Naz_new = az_new.size
    Nza_new = za_new.size

    # RectBivariateSpline requires strictly increasing axes
    if not (np.all(np.diff(za_old) > 0) and np.all(np.diff(az_old) > 0)):
        raise ValueError("za_old and az_old must be strictly increasing for RectBivariateSpline.")
    if not (np.all(np.diff(za_new) > 0) and np.all(np.diff(az_new) > 0)):
        raise ValueError("za_new and az_new must be strictly increasing.")

    beam_resampled = beam.copy()
    new_data = np.empty(
        (beam.Naxes_vec, beam.Nfeeds, beam.Nfreqs, Nza_new, Naz_new),
        dtype=beam.data_array.dtype,
    )

    # Loop over (vec, feed, freq) and spline-interpolate each 2D slice
    for ivec in range(beam.Naxes_vec):
        for ifeed in range(beam.Nfeeds):
            for ifreq in range(beam.Nfreqs):
                slice_2d = beam.data_array[ivec, ifeed, ifreq, :, :]  # (Nza_old, Naz_old)

                spline_real = RectBivariateSpline(
                    za_old, az_old, slice_2d.real,
                    kx=kx, ky=ky
                )
                spline_imag = RectBivariateSpline(
                    za_old, az_old, slice_2d.imag,
                    kx=kx, ky=ky
                )

                vals_real = spline_real(za_new, az_new)  # (Nza_new, Naz_new)
                vals_imag = spline_imag(za_new, az_new)

                new_data[ivec, ifeed, ifreq, :, :] = (
                    vals_real + 1j * vals_imag
                ).astype(beam.data_array.dtype)

    # Update grid metadata
    beam_resampled.axis1_array = az_new
    beam_resampled.axis2_array = za_new
    beam_resampled.Naxes1 = Naz_new
    beam_resampled.Naxes2 = Nza_new

    if hasattr(beam_resampled, "Npixels"):
        beam_resampled.Npixels = Naz_new * Nza_new

    beam_resampled.data_array = new_data

    beam_resampled.history += (
        f"\nResampled az/za grid to Nza={Nza_new}, Naz={Naz_new} "
        f"using RectBivariateSpline(kx={kx}, ky={ky})."
    )

    return beam_resampled


3. Helper function : resample to general template

In [ ]:
def resample_uvbeam_to_template(
    beam_to_resample: UVBeam,
    template_beam: UVBeam,
    kx: int = 3,
    ky: int = 3,
) -> UVBeam:
    """
    Resample `beam_to_resample` onto the az/za grid of `template_beam`.

    Typical use:
    - Start from a coarse beam (template_beam)
    - Upsample it (beam_fine)
    - Rotate beam_fine (beam_rot)
    - Then call this to go back to the original coarse grid.
    """
    if template_beam.pixel_coordinate_system != "az_za":
        raise ValueError("template_beam must have pixel_coordinate_system == 'az_za'")
    if beam_to_resample.pixel_coordinate_system != "az_za":
        raise ValueError("beam_to_resample must have pixel_coordinate_system == 'az_za'")

    za_new = template_beam.axis2_array
    az_new = template_beam.axis1_array

    beam_resampled = resample_uvbeam_azza(
        beam_to_resample,
        za_new=za_new,
        az_new=az_new,
        kx=kx,
        ky=ky,
    )

    beam_resampled.history += (
        f"\nResampled to match template beam grid "
        f"(Nza={template_beam.Naxes2}, Naz={template_beam.Naxes1})."
    )

    return beam_resampled


4. Helper function : beam rotator

In [ ]:
import string
from pyuvdata import UVBeam

def airy_rotated_to_healpix_efield(
    beam_rot: UVBeam,
    nside: int | None = None,
    outfilename: str | None = None,
    clobber: bool = True,
) -> UVBeam:
    """
    Convert a rotated Airy UVBeam (efield, az_za) to a HEALPix efield UVBeam.

    Parameters
    ----------
    beam_rot : UVBeam
        Rotated beam, must have pixel_coordinate_system == 'az_za'
        and beam_type == 'efield'.
    nside : int or None
        Healpix nside. If None, pyuvdata will choose a suitable nside
        based on your az/za resolution.
    outfilename : str or None
        If given, write a beamfits file to this path.
    clobber : bool
        Overwrite existing file if True.

    Returns
    -------
    hpx_beam : UVBeam
        New UVBeam in HEALPix, efield format.
    """
    uvb = beam_rot.copy()

    # Sanity checks
    if uvb.pixel_coordinate_system != "az_za":
        raise ValueError("Expected az_za beam as input.")
    if uvb.beam_type != "efield":
        raise ValueError("Expected efield beam as input.")

    # Required interpolation scheme for az_za → healpix
    uvb.interpolation_function = "az_za_simple"

    # Convert to healpix efield (no 'order' kwarg in this pyuvdata version)
    if nside is None:
        hpx_beam = uvb.to_healpix(inplace=False)
    else:
        hpx_beam = uvb.to_healpix(nside=nside, inplace=False)

    # Just to be explicit:
    assert hpx_beam.beam_type == "efield"
    assert hpx_beam.pixel_coordinate_system == "healpix"

    # --- Clean history so FITS (ASCII-only) is happy ---
    def force_ascii(s: str) -> str:
        return "".join(ch if ch in string.printable else "?" for ch in s)

    hpx_beam.history = force_ascii(hpx_beam.history)
    hpx_beam.history += (
        "\nConverted rotated Airy beam from az_za to healpix "
        f"(beam_type=efield, nside={getattr(hpx_beam, 'nside', 'unknown')})."
    )
    hpx_beam.history = force_ascii(hpx_beam.history)

    # Optionally write out
    if outfilename is not None:
        hpx_beam.write_beamfits(outfilename, clobber=clobber)

    return hpx_beam


5. Helper function : beam rotation sub functions

In [ ]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator

# --- your rotation helpers (unchanged) ---

def rotation_matrix_y(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[ c, 0, s],
                     [ 0, 1, 0],
                     [-s, 0, c]])

def rotation_matrix_z(phi):
    c, s = np.cos(phi), np.sin(phi)
    return np.array([[ c, -s, 0],
                     [ s,  c, 0],
                     [ 0,  0, 1]])

def rotation_matrix(theta0, phi0):
    """Composite rotation R = Rz(phi0) @ Ry(theta0)."""
    Rz = rotation_matrix_z(phi0)
    Ry = rotation_matrix_y(theta0)
    return Rz @ Ry  # column-vector convention

# --- az/za <-> Cartesian ---

def azza_to_cart(az, za, r=1.0):
    """
    (az, za) -> (x,y,z) on sphere of radius r.
    az, za can be scalars or arrays.
    """
    az = np.asarray(az, dtype=float)
    za = np.asarray(za, dtype=float)
    x = r * np.sin(za) * np.cos(az)
    y = r * np.sin(za) * np.sin(az)
    z = r * np.cos(za)
    return np.stack((x, y, z), axis=-1)

def cart_to_azza(xyz):
    """
    (x,y,z) -> (r, az, za). xyz shape (...,3).
    """
    xyz = np.asarray(xyz, dtype=float)
    x, y, z = xyz[..., 0], xyz[..., 1], xyz[..., 2]
    r = np.sqrt(x*x + y*y + z*z)
    # avoid tiny numerical issues
    z_clipped = np.clip(z / np.where(r == 0, 1.0, r), -1.0, 1.0)
    za = np.arccos(z_clipped)
    az = np.mod(np.arctan2(y, x), 2*np.pi)
    return r, az, za


6. Beam rotator

In [ ]:
from pyuvdata import UVBeam
from scipy.interpolate import RegularGridInterpolator

def rotate_uvbeam_pattern(beam: UVBeam, theta0, phi0, degrees=True) -> UVBeam:
    """
    Rotate a UVBeam's pattern by angles (theta0, phi0) w.r.t. the zenith.

    - beam.pixel_coordinate_system must be 'az_za'
    - data_array shape assumed (2, 2, Nfreq, Nza, Naz)

    theta0: rotation about +y (tilt away from zenith)
    phi0  : rotation about +z (spin in azimuth)
    (angles measured in *beam frame*; see note below)

    Returns a *new* UVBeam with rotated data_array (metadata copied).
    """
    if beam.pixel_coordinate_system != 'az_za':
        raise ValueError("This helper expects beam.pixel_coordinate_system == 'az_za'")

    if degrees:
        theta0 = np.deg2rad(theta0)
        phi0   = np.deg2rad(phi0)

    # rotation matrix for physical beam orientation
    R = rotation_matrix(theta0, phi0)       # R acts on column vectors

    # original grids
    az_array = beam.axis1_array            # shape (Naz,)
    za_array = beam.axis2_array            # shape (Nza,)
    Naz = az_array.size
    Nza = za_array.size

    # 2D grid of OUT directions (where the rotated beam is evaluated)
    ZA_out, AZ_out = np.meshgrid(za_array, az_array, indexing='ij')  # (Nza, Naz)

    # convert OUT directions -> xyz
    pts_out = azza_to_cart(AZ_out, ZA_out, r=1.0)    # (Nza, Naz, 3)

    # We want B_rot(n_out) = B_orig(R^{-1} n_out).
    # With our convention v' = R v (columns), R^{-1} = R^T.
    # In row-vector form: v_row_out = v_row_in @ R.T → v_row_in = v_row_out @ R.
    pts_out_flat = pts_out.reshape(-1, 3)                # (N,3) row-vectors
    pts_in_flat  = pts_out_flat @ R                      # apply inverse rotation
    _, AZ_in_flat, ZA_in_flat = cart_to_azza(pts_in_flat)

    # wrap / clip into the original grid domain
    AZ_in_flat = np.mod(AZ_in_flat, 2*np.pi)
    ZA_in_flat = np.clip(ZA_in_flat, 0.0, np.pi)

    sample_points = np.stack((ZA_in_flat, AZ_in_flat), axis=-1)  # (N,2)

    # prepare new beam as a copy
    new_beam = beam.copy()
    new_data = np.empty_like(beam.data_array)

    # RegularGridInterpolator expects axes in same order as data: (za, az)
    za_grid = za_array
    az_grid = az_array

    interp_type = 'linear'  # 'linear' or 'cubic'
    
    if interp_type == 'linear':
        # Loop over (vec, feed, freq) and interpolate the 2D slice
        for ivec in range(beam.Naxes_vec):
            for ifeed in range(beam.Nfeeds):
                for ifreq in range(beam.Nfreqs):
                    slice_2d = beam.data_array[ivec, ifeed, ifreq, :, :]  # (Nza, Naz)

                    interp = RegularGridInterpolator(
                        (za_grid, az_grid),
                        slice_2d,
                        bounds_error=False,
                        fill_value=0.0,   # outside original FOV → 0 response
                    )

                    vals_flat = interp(sample_points)           # (N,)
                    new_slice = vals_flat.reshape(Nza, Naz)     # back to (Nza, Naz)
                    new_data[ivec, ifeed, ifreq, :, :] = new_slice
                
    elif interp_type == 'cubic':
        from scipy.interpolate import RectBivariateSpline
        ZA_in_flat = sample_points[:, 0]
        AZ_in_flat = sample_points[:, 1]

        # Loop over (vec, feed, freq) and interpolate the 2D slice with cubic splines
        for ivec in range(beam.Naxes_vec):
            for ifeed in range(beam.Nfeeds):
                for ifreq in range(beam.Nfreqs):
                    slice_2d = beam.data_array[ivec, ifeed, ifreq, :, :]  # (Nza, Naz)

                    # Build separate splines for real and imaginary parts
                    spline_real = RectBivariateSpline(
                        za_grid, az_grid, slice_2d.real,
                        kx=2, ky=2   # cubic in both directions
                    )
                    spline_imag = RectBivariateSpline(
                        za_grid, az_grid, slice_2d.imag,
                        kx=2, ky=2
                    )

                    vals_real = spline_real.ev(ZA_in_flat, AZ_in_flat)
                    vals_imag = spline_imag.ev(ZA_in_flat, AZ_in_flat)
                    vals_flat = vals_real + 1j * vals_imag

                    new_slice = vals_flat.reshape(Nza, Naz)
                    new_data[ivec, ifeed, ifreq, :, :] = new_slice

    new_beam.data_array = new_data
    # history note
    new_beam.history += f"\nRotated pattern by theta={theta0:.4f} rad, phi={phi0:.4f} rad."
    return new_beam


7. Airy Beam Generator :


Contains : 


    a. Rotation toggle
    b. Achromaticity toggle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyuvdata import UVBeam
from scipy.special import j1
from copy import deepcopy

def create_airy_uvbeam(
    diameter: float,
    freq_array: np.ndarray,
    az_array: np.ndarray = None,
    za_array: np.ndarray = None,
    telescope_name: str = 'HERA',
    feed_name: str = 'airy',
    below_horizon: str = 'exponential',
    suppression_floor_db: float = -40.0,
    decay_rate_db_per_deg: float = 0.4,
    decay_start_za_deg: float = 90.0,
    reference_freq_hz: float | None = None,
) -> UVBeam:
    """
    Create an Airy beam pattern and store in UVBeam format.
    If reference_freq_hz is set, the Airy pattern is computed once at that
    frequency and reused for every channel.

    Parameters
    ----------
    diameter : float
        Dish diameter in meters
    freq_array : np.ndarray
        Frequency array in Hz
    decay_rate_db_per_deg : float
        Decay rate in dB per degree past decay_start_za_deg
    decay_start_za_deg : float
        Zenith angle (degrees) where exponential decay begins.
        Default 90.0 (horizon). Use smaller values to start decay earlier.
    """

    if az_array is None:
        az_array = np.deg2rad(np.arange(0, 360, 1).astype(float))
    if za_array is None:
        za_array = np.deg2rad(np.arange(0, 181, 1).astype(float))

    freq_array = np.asarray(freq_array, dtype=float)

    c = 299792458.0
    n_freq = len(freq_array)
    n_az = len(az_array)
    n_za = len(za_array)

    if reference_freq_hz is None:
        shape_freq_array = freq_array
    else:
        shape_freq_array = np.full(n_freq, float(reference_freq_hz))

    k_array = 2 * np.pi * shape_freq_array / c
    za_mesh, k_mesh = np.meshgrid(za_array, k_array, indexing='ij')
    x_vals = (diameter / 2.0) * np.sin(za_mesh) * k_mesh
#   print("x_vals :", x_vals.shape, x_vals)

    with np.errstate(divide='ignore', invalid='ignore'):
        airy_pattern = np.where(np.abs(x_vals) < 1e-10, 1.0, 2.0 * j1(x_vals) / x_vals)
    #   print("airy_pattern initial :", airy_pattern.shape, airy_pattern)

    # Exponential decay starting at decay_start_za_deg
    if below_horizon == 'exponential':
        for i_za, za in enumerate(za_array):
            za_deg = np.rad2deg(za)
            if za_deg > decay_start_za_deg:
                degrees_past_start = za_deg - decay_start_za_deg
                suppression_db = -decay_rate_db_per_deg * degrees_past_start
                suppression_db = max(suppression_db, suppression_floor_db)
                attenuation = 10 ** (suppression_db / 20)
                airy_pattern[i_za, :] *= attenuation

    efield_component = airy_pattern / np.sqrt(2)

    data_array = np.zeros((2, 2, n_freq, n_za, n_az), dtype=np.complex128)
    efield_transposed = efield_component.T

    for i_az in range(n_az):
        data_array[0, 0, :, :, i_az] = efield_transposed
        data_array[0, 1, :, :, i_az] = efield_transposed
        data_array[1, 0, :, :, i_az] = efield_transposed
        data_array[1, 1, :, :, i_az] = efield_transposed

    basis_vector_array = np.zeros((2, 2, n_za, n_az))
    basis_vector_array[0, 0, :, :] = 1.0
    basis_vector_array[1, 1, :, :] = 1.0

    beam = UVBeam()

    beam.pixel_coordinate_system = 'az_za'
    beam.beam_type = 'efield'
    beam.data_normalization = 'peak'
    beam.antenna_type = 'simple'

    beam.telescope_name = telescope_name
    beam.feed_name = feed_name
    beam.feed_version = '1.0'

    if reference_freq_hz is None:
        beam.model_name = f'Airy disk (D={diameter}m)'
    else:
        beam.model_name = f'Airy disk (D={diameter}m, fixed at {reference_freq_hz/1e6:.1f} MHz)'

    beam.model_version = '1.0'

    if reference_freq_hz is None:
        beam.history = (f'Analytic Airy beam D={diameter}m. '
                        f'Exponential decay {decay_rate_db_per_deg} dB/deg starting at ZA={decay_start_za_deg} deg.')
    else:
        beam.history = (f'Analytic Airy beam D={diameter}m frozen at {reference_freq_hz/1e6:.1f} MHz. '
                        f'Exponential decay {decay_rate_db_per_deg} dB/deg starting at ZA={decay_start_za_deg} deg.')

    beam.Naxes_vec = 2
    beam.Ncomponents_vec = 2
    beam.Nfeeds = 2
    beam.Nfreqs = n_freq
    beam.Naxes1 = n_az
    beam.Naxes2 = n_za

    beam.axis1_array = az_array
    beam.axis2_array = za_array
    beam.freq_array = freq_array

    beam.feed_array = np.array(['x', 'y'])
    beam.feed_angle = np.array([np.pi/2, 0.0])

    beam.data_array = data_array
    beam.basis_vector_array = basis_vector_array
    beam.bandpass_array = np.ones(n_freq, dtype=float)

    beam.check()

    return beam


# ============================================
# PARAMETERS - CHANGE THESE AS NEEDED
# ============================================
decay_rate = 0.3           # dB per degree
decay_start_za = 70.0      # zenith angle where decay starts (degrees)
diameter = 14.0            # dish diameter (meters)
freq_array = np.linspace(45e6, 250e6, 206)
suppression_floor_db = -40.0   # floor for the exponential decay (dB)

# --- Achromaticity toggle ---
# use_ref_freq=True  -> freeze the beam shape at a single reference frequency
# use_ref_freq=False -> normal chromatic Airy beam
use_ref_freq = True
ref_freq_hz = 80e6         # reference frequency in Hz (80e6 Hz = 80 MHz)
ref_freq = ref_freq_hz if use_ref_freq else None

# --- Rotation toggle ---
rotate = False              # True -> tilt beam off zenith; False -> leave zenith-pointing
theta0_deg = 2.0           # tilt away from zenith (only used if rotate=True)
phi0_deg   = 270.0         # rotate in azimuth      (only used if rotate=True)

# ============================================
# CREATE BEAM
# ============================================
if ref_freq is None:
    print("Creating Airy beam:")
else:
    print("Creating frequency-independent Airy beam:")
    print(f"  Reference frequency: {ref_freq/1e6:.1f} MHz")
print(f"  Diameter: {diameter} m")
print(f"  Decay rate: {decay_rate} dB/deg")
print(f"  Decay starts at: ZA = {decay_start_za} deg")

beam = create_airy_uvbeam(
    diameter=diameter,
    freq_array=freq_array,
    telescope_name='HERA',
    feed_name='airy_exponential',
    below_horizon='exponential',
    suppression_floor_db=suppression_floor_db,
    decay_rate_db_per_deg=decay_rate,
    decay_start_za_deg=decay_start_za,
    reference_freq_hz=ref_freq,
)
slice_first = beam.data_array[0, 0, 0, :, 0].real
slice_last = beam.data_array[0, 0, -1, :, 0].real
print(f"Max abs diff between first and last freq slice: {np.max(np.abs(slice_first - slice_last)):.2e}")

# Keep an un-rotated copy (used by the 3D viewer cell for the left-hand panel)
unrot_beam = beam.copy()

# ============================================
# ROTATE BEAM TO POINTING DIRECTION (optional)
# ============================================
fine_rot_beam = None  # only populated when rotate=True

if rotate:
    print("Rotating beam")

    # Upsample only when we actually rotate (avoids wasted compute otherwise)
    beam_fine = upsample_uvbeam_azza(
        beam,
        upsample_factor_za=2,   # or 3, etc.
        upsample_factor_az=2,
        kx=3,
        ky=3,
    )

    # Rotate the (fine) zenith-pointing Airy beam
    fine_rot_beam = rotate_uvbeam_pattern(beam_fine, theta0_deg, phi0_deg, degrees=True)

    # Check: where is the new peak?
    # (rough check by brute-force argmax on one freq, one feed/component)
    freq_idx = 100
    slice_power = np.abs(fine_rot_beam.data_array[0, 0, freq_idx])**2
    iz, iaz = np.unravel_index(np.argmax(slice_power), slice_power.shape)
    print("Peak at za' [deg], az' [deg] =",
        np.rad2deg(fine_rot_beam.axis2_array[iz]),
        np.rad2deg(fine_rot_beam.axis1_array[iaz]))

    # Downsample rotated beam back to the original grid
    beam_rot_coarse = resample_uvbeam_to_template(
        fine_rot_beam,
        template_beam=beam,
        kx=3,
        ky=3,
    )

    beam = deepcopy(beam_rot_coarse)


# ============================================
# FILENAME TAGS (single source of truth for both .fits and .png)
# ============================================
freq_tag = f"_freqconst_ref{ref_freq/1e6:.0f}MHz" if ref_freq is not None else ""
rot_tag = f"_rttd_za_az_{theta0_deg}_{phi0_deg}" if rotate else ""

# Save with parameters in filename
save_out = "/home/herastore02-1/HERA_Validation_rchandra/"

base_name = f'airy_beam_{diameter}m{freq_tag}_decay_{decay_rate}dBdeg_start_{decay_start_za}deg{rot_tag}'
filename = save_out + base_name + '.fits'

beam.write_beamfits(filename, clobber=True)
print(f"\nSaved: {filename}")

if not rotate:
    # Choose nside ~ 64 for ~1 deg resolution; adjust if you want finer/coarser
    healpix_filename = save_out + base_name + '_healpix.fits'

    beam_hpx = airy_rotated_to_healpix_efield(
        beam,
        nside=64,
        outfilename=healpix_filename,
        clobber=True,
    )

    print("Wrote HEALPix efield beam to:", healpix_filename)
    print("beam_type:", beam_hpx.beam_type)
    print("pixel_coordinate_system:", beam_hpx.pixel_coordinate_system)
    print("nside:", getattr(beam_hpx, "nside", None))



# ============================================
# CHECK BASIC METADATA
# ============================================
print("\n" + "=" * 60)
print("BEAM METADATA")
print("=" * 60)
print(f"Beam is valid:       {beam.check()}")
print(f"Telescope Name:      {beam.telescope_name}")
print(f"Feed Name:           {beam.feed_name}")
print(f"Model Name:          {beam.model_name}")
print(f"Beam type:           {beam.beam_type}")
print(f"Coordinate system:   {beam.pixel_coordinate_system}")
print(f"Data normalization:  {beam.data_normalization}")
print(f"Data shape:          {beam.data_array.shape}")
print(f"Data dtype:          {beam.data_array.dtype}")
print(f"Freq range:          {beam.freq_array.min()/1e6:.1f} - {beam.freq_array.max()/1e6:.1f} MHz")
print(f"Num frequencies:     {beam.Nfreqs}")
print(f"Naxes1 (azimuth):    {beam.Naxes1}")
print(f"Naxes2 (zenith):     {beam.Naxes2}")
print(f"Feed array:          {beam.feed_array}")
print(f"Feed angle:          {beam.feed_angle}")
print(f"NaNs in beam:        {np.isnan(beam.data_array).sum()}")
print(f"Infs in beam:        {np.isinf(beam.data_array).sum()}")
print(f"History:             {beam.history}")


# ============================================
# PLOT BEAM PATTERN (the final saved beam)
# ============================================
def plot_airy_za_cut(plot_beam, png_path, title_suffix=""):
    freq_idx = np.argmin(np.abs(freq_array - 150e6))
    actual_freq = freq_array[freq_idx] / 1e6
    za_deg = np.rad2deg(plot_beam.axis2_array)

    e_theta = plot_beam.data_array[0, 0, freq_idx, :, 0]
    e_phi = plot_beam.data_array[1, 0, freq_idx, :, 0]
    power = np.abs(e_theta)**2 + np.abs(e_phi)**2
    power_db = 10 * np.log10(power / np.nanmax(power) + 1e-10)

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(za_deg, power_db, 'b-', linewidth=2, label=f'Airy beam ({decay_rate} dB/deg decay)')
    ax.axvline(decay_start_za, color='red', linestyle='--', linewidth=1.5,
                label=f'Decay start (ZA={decay_start_za} deg)')
    ax.axhline(suppression_floor_db, color='gray', linestyle=':', alpha=0.7,
                label=f'Floor ({suppression_floor_db:.0f} dB)')

    ax.set_xlabel('Zenith Angle (degrees)', fontsize=12)
    ax.set_ylabel('Normalized Power (dB)', fontsize=12)
    ax.set_title(f'Airy Beam ({diameter}m, {decay_rate} dB/deg decay from ZA={decay_start_za} deg) '
                 f'at {actual_freq:.1f} MHz{title_suffix}',
                fontsize=14)
    ax.set_xlim(0, 180)
    ax.set_ylim(-60, 5)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    ax.fill_between([decay_start_za, 180], -60, 5, alpha=0.1, color='gray')
    ax.text(decay_start_za/2, -55, 'No decay', ha='center', fontsize=10, style='italic')
    ax.text((decay_start_za + 180)/2, -55, 'Exponential decay', ha='center', fontsize=10, style='italic')

    plt.tight_layout()
    plt.savefig(png_path, dpi=150)
    plt.show()
    return png_path

# Plot the final beam that was actually saved to FITS
final_png = base_name + '.png'
plot_airy_za_cut(beam, final_png)
print(f"Plot saved: {final_png}")

# Diagnostic plot of the high-resolution rotated beam (distinct filename, no clobber)
if rotate:
    fine_png = base_name + '_finegrid.png'
    plot_airy_za_cut(fine_rot_beam, fine_png, title_suffix=" [fine grid]")
    print(f"Fine-grid plot saved: {fine_png}")

print("Done!")


8. 3D beam viewer

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# ============================================
# 3D PLOTLY VIEWER — Normal vs Rotated Airy Beam
# ============================================

def beam_to_3d_surface(uvbeam, freq_array, target_freq_mhz=150.0, min_db=-40.0):
    """Convert a UVBeam to 3D surface data (X, Y, Z, color, hover)."""
    az = uvbeam.axis1_array   # (Naz,) radians
    za = uvbeam.axis2_array   # (Nza,) radians

    freq_idx = np.argmin(np.abs(freq_array - target_freq_mhz * 1e6))

    e_theta = uvbeam.data_array[0, 0, freq_idx, :, :]  # (Nza, Naz)
    e_phi   = uvbeam.data_array[1, 0, freq_idx, :, :]

    power = np.abs(e_theta)**2 + np.abs(e_phi)**2
    power_db = 10 * np.log10(power / np.nanmax(power) + 1e-30)
    power_db_clipped = np.clip(power_db, min_db, 0.0)

    # Map dB to radius: [min_db, 0] -> [0, 1]
    radius = (power_db_clipped - min_db) / (0.0 - min_db)

    AZ, ZA = np.meshgrid(az, za)
    X = radius * np.sin(ZA) * np.cos(AZ)
    Y = radius * np.sin(ZA) * np.sin(AZ)
    Z = radius * np.cos(ZA)

    AZ_deg = np.rad2deg(AZ)
    ZA_deg = np.rad2deg(ZA)

    hover_text = [[
        f"Az: {AZ_deg[i, j]:.1f}°<br>"
        f"ZA: {ZA_deg[i, j]:.1f}°<br>"
        f"Power: {power_db_clipped[i, j]:.2f} dB"
        for j in range(AZ_deg.shape[1])]
        for i in range(AZ_deg.shape[0])
    ]
    return X, Y, Z, power_db_clipped, hover_text, freq_array[freq_idx] / 1e6

# --- Build surfaces ---
target_mhz = 150.0

# Resolve inputs robustly: works right after the generator cell, after loading
# beams from FITS, or with rotation off. No dependence on a specific run order.
if 'unrot_beam' in globals():
    left_beam = unrot_beam
elif 'beam' in globals():
    left_beam = beam
else:
    raise NameError("No beam found: define `unrot_beam` or `beam` before running the viewer.")
freq_for_lookup = freq_array if 'freq_array' in globals() else left_beam.freq_array
_rotated_view = bool(globals().get('rotate', False)) and (globals().get('fine_rot_beam') is not None)
_diam       = globals().get('diameter', 'NA')
_decay      = globals().get('decay_rate', 'NA')
_decaystart = globals().get('decay_start_za', 'NA')

X1, Y1, Z1, C1, H1, actual_mhz = beam_to_3d_surface(left_beam, freq_for_lookup, target_mhz)

if _rotated_view:
    X2, Y2, Z2, C2, H2, _ = beam_to_3d_surface(fine_rot_beam, freq_for_lookup, target_mhz)
    right_title = f"Rotated Airy (θ₀={theta0_deg}°, φ₀={phi0_deg}°)"
else:
    X2, Y2, Z2, C2, H2 = X1, Y1, Z1, C1, H1
    right_title = "Airy Beam [rotation OFF]"

# --- Side-by-side 3D subplots ---
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'surface'}, {'type': 'surface'}]],
    subplot_titles=[f"Normal Airy ({_diam}m)", right_title],
    horizontal_spacing=0.05,
)

fig.add_trace(
    go.Surface(
        x=X1, y=Y1, z=Z1,
        surfacecolor=C1,
        colorscale='Viridis',
        colorbar=dict(
            title=dict(text='dB', font=dict(size=12)),
            ticksuffix=' dB', x=0.44, len=0.75,
        ),
        hoverinfo='text', text=H1,
        name='Normal',
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Surface(
        x=X2, y=Y2, z=Z2,
        surfacecolor=C2,
        colorscale='Viridis',
        colorbar=dict(
            title=dict(text='dB', font=dict(size=12)),
            ticksuffix=' dB', x=1.0, len=0.75,
        ),
        hoverinfo='text', text=H2,
        name='Rotated',
    ),
    row=1, col=2,
)

scene_kw = dict(
    xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
    aspectmode='data',
)
fig.update_layout(
    scene=scene_kw,
    scene2=scene_kw,
    title=dict(
        text=(f"3D Airy Beam @ {actual_mhz:.1f} MHz  |  "
              f"D={_diam}m, decay={_decay} dB/deg from ZA={_decaystart}°"),
        font=dict(size=15),
    ),
    width=1500, height=700,
    margin=dict(l=10, r=10, t=60, b=10),
)

fig.show()
print("Done!")

9. Gaussian beam generator

In [ ]:
import numpy as np
from pyuvdata import UVBeam


def create_gaussian_uvbeam(
    sigma_deg: float = None,
    diameter: float = None,
    freq_array: np.ndarray = None,
    az_array: np.ndarray = None,
    za_array: np.ndarray = None,
    telescope_name: str = "HERA",
    feed_name: str = "gaussian",
    below_horizon: str = "exponential",
    suppression_floor_db: float = -40.0,
    decay_rate_db_per_deg: float = 0.4,
    decay_start_za_deg: float = 90.0,
) -> UVBeam:
    """
    Create a Gaussian beam pattern and store in UVBeam format.

    The beam voltage pattern is  E(za) = exp( -za^2 / (2 sigma^2) ).

    Sigma can be specified in two ways (supply exactly one):

    1. **sigma_deg** - a fixed angular width (degrees), constant across
       frequency.  Useful for quick tests.
    2. **diameter** - dish diameter in metres.  Sigma is then derived
       from the diffraction limit at each frequency:

           FWHM(f) = 1.02 * lambda / D          (radians)
           sigma(f) = FWHM / (2 sqrt(2 ln 2))   (radians)

       so the beam narrows at higher frequencies, as expected.

    Parameters
    ----------
    sigma_deg : float or None
        Fixed Gaussian sigma in degrees (frequency-independent).
    diameter : float or None
        Dish diameter in metres (gives frequency-dependent sigma).
    freq_array : np.ndarray
        Frequencies in Hz.
    az_array, za_array : np.ndarray or None
        Azimuth / zenith-angle grids in radians.  Defaults to 1-deg steps.
    below_horizon : str
        'exponential' to apply decaying envelope past `decay_start_za_deg`,
        or 'none' to leave the Gaussian tail untouched.
    decay_rate_db_per_deg : float
        Decay rate in dB per degree past `decay_start_za_deg`.
    decay_start_za_deg : float
        ZA (degrees) where exponential suppression begins.
    suppression_floor_db : float
        Minimum suppression in dB (floor).

    Returns
    -------
    beam : UVBeam
        E-field UVBeam on an az_za pixel grid.
    """
    # ---- Input validation ------------------------------------------------
    if (sigma_deg is None) == (diameter is None):
        raise ValueError("Supply exactly one of `sigma_deg` or `diameter`.")
    if freq_array is None:
        raise ValueError("`freq_array` is required.")

    if az_array is None:
        az_array = np.deg2rad(np.arange(0, 360, 1).astype(float))
    if za_array is None:
        za_array = np.deg2rad(np.arange(0, 181, 1).astype(float))

    freq_array = np.asarray(freq_array, dtype=float)
    c = 299792458.0
    n_freq = len(freq_array)
    n_az = len(az_array)
    n_za = len(za_array)

    # ---- Build sigma array (one per frequency) ---------------------------
    if sigma_deg is not None:
        # Constant sigma across frequency
        sigma_rad = np.full(n_freq, np.deg2rad(sigma_deg))
    else:
        # Diffraction-limited:  FWHM = 1.02 * lambda / D
        wavelengths = c / freq_array
        fwhm_rad = 1.02 * wavelengths / diameter
        sigma_rad = fwhm_rad / (2.0 * np.sqrt(2.0 * np.log(2.0)))

    # ---- Gaussian pattern on (za, freq) grid ----------------------------
    # za_mesh shape: (n_za, n_freq),  sigma_mesh same
    za_mesh, sigma_mesh = np.meshgrid(za_array, sigma_rad, indexing="ij")
    gaussian_pattern = np.exp(-0.5 * (za_mesh / sigma_mesh) ** 2)  # (n_za, n_freq)

    # ---- Below-horizon suppression (identical to Airy version) -----------
    if below_horizon == "exponential":
        for i_za, za in enumerate(za_array):
            za_deg = np.rad2deg(za)
            if za_deg > decay_start_za_deg:
                degrees_past = za_deg - decay_start_za_deg
                suppression_db = max(-decay_rate_db_per_deg * degrees_past,
                                     suppression_floor_db)
                gaussian_pattern[i_za, :] *= 10.0 ** (suppression_db / 20.0)
    if below_horizon != "exponential":
        print("below horizon suppression disabled ", below_horizon)

    # ---- Pack into UVBeam ------------------------------------------------
    efield_component = gaussian_pattern / np.sqrt(2)
    efield_transposed = efield_component.T  # (n_freq, n_za)

    data_array = np.zeros((2, 2, n_freq, n_za, n_az), dtype=np.complex128)
    for i_az in range(n_az):
        data_array[0, 0, :, :, i_az] = efield_transposed
        data_array[0, 1, :, :, i_az] = efield_transposed
        data_array[1, 0, :, :, i_az] = efield_transposed
        data_array[1, 1, :, :, i_az] = efield_transposed

    basis_vector_array = np.zeros((2, 2, n_za, n_az))
    basis_vector_array[0, 0, :, :] = 1.0
    basis_vector_array[1, 1, :, :] = 1.0

    beam = UVBeam()
    beam.pixel_coordinate_system = "az_za"
    beam.beam_type = "efield"
    beam.data_normalization = "peak"
    beam.antenna_type = "simple"

    beam.telescope_name = telescope_name
    beam.feed_name = feed_name
    beam.feed_version = "1.0"
    if sigma_deg is not None:
        beam.model_name = f"Gaussian (sigma={sigma_deg:.2f} deg, fixed)"
    else:
        beam.model_name = f"Gaussian (D={diameter}m, diffraction-limited)"
    beam.model_version = "1.0"

    sigma_info = (f"sigma_deg={sigma_deg}" if sigma_deg is not None
                  else f"D={diameter}m, sigma(150 MHz)="
                       f"{np.rad2deg(sigma_rad[np.argmin(np.abs(freq_array-150e6))]):.2f} deg")
    beam.history = (
        f"Analytic Gaussian beam ({sigma_info}). "
        f"Exponential decay {decay_rate_db_per_deg} dB/deg "
        f"starting at ZA={decay_start_za_deg} deg."
    )

    beam.Naxes_vec = 2
    beam.Ncomponents_vec = 2
    beam.Nfeeds = 2
    beam.Nfreqs = n_freq
    beam.Naxes1 = n_az
    beam.Naxes2 = n_za

    beam.axis1_array = az_array
    beam.axis2_array = za_array
    beam.freq_array = freq_array

    beam.feed_array = np.array(["x", "y"])
    beam.feed_angle = np.array([np.pi / 2, 0.0])

    beam.data_array = data_array
    beam.basis_vector_array = basis_vector_array
    beam.bandpass_array = np.ones(n_freq, dtype=float)

    beam.check()
    return beam

10. Gaussian beam generation cell

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

# ============================================
# PARAMETERS - CHANGE THESE AS NEEDED
# ============================================
decay_rate = 0.3           # dB per degree
decay_start_za = 70.0      # zenith angle where decay starts (degrees)
diameter = 7.0            # dish diameter (meters) — used for freq-dependent sigma
sigma_deg = None           # set to e.g. 10.0 for fixed sigma; leave None to use diameter
freq_array = np.linspace(45e6, 250e6, 206)

# ============================================
# CREATE GAUSSIAN BEAM
# ============================================
if sigma_deg is not None:
    print(f"Creating Gaussian beam (fixed sigma):")
    print(f"  sigma: {sigma_deg} deg")
else:
    print(f"Creating Gaussian beam (diffraction-limited):")
    print(f"  Diameter: {diameter} m")
print(f"  Decay rate: {decay_rate} dB/deg")
print(f"  Decay starts at: ZA = {decay_start_za} deg")

gauss_beam = create_gaussian_uvbeam(
    sigma_deg=sigma_deg,
    diameter=diameter,
    freq_array=freq_array,
    telescope_name='HERA',
    feed_name='gaussian_exponential',
    below_horizon='exponential',
    decay_rate_db_per_deg=decay_rate,
    decay_start_za_deg=decay_start_za,
)

unrot_gauss_beam = gauss_beam.copy()  # keep unrotated copy for comparison

gauss_beam_fine = upsample_uvbeam_azza(
    gauss_beam,
    upsample_factor_za=2,
    upsample_factor_az=2,
    kx=3,
    ky=3,
)

# ============================================
# ROTATE BEAM TO POINTING DIRECTION (optional)
# ============================================
rotate_gauss = False  # Set to True to rotate beam
if rotate_gauss:
    print("Rotating Gaussian beam")

    theta0_deg = 2.0   # tilt away from zenith
    phi0_deg   = 270.0  # rotate in azimuth

    fine_rot_gauss_beam = rotate_uvbeam_pattern(gauss_beam_fine, theta0_deg, phi0_deg, degrees=True)

    # Check: where is the new peak?
    freq_idx = 100
    slice_power = np.abs(fine_rot_gauss_beam.data_array[0, 0, freq_idx])**2
    iz, iaz = np.unravel_index(np.argmax(slice_power), slice_power.shape)
    print("Peak at za' [deg], az' [deg] =",
        np.rad2deg(fine_rot_gauss_beam.axis2_array[iz]),
        np.rad2deg(fine_rot_gauss_beam.axis1_array[iaz]))

    # Downsample rotated beam back to original grid
    gauss_beam_rot_coarse = resample_uvbeam_to_template(
        fine_rot_gauss_beam,
        template_beam=gauss_beam,
        kx=3,
        ky=3,
    )

    gauss_beam = deepcopy(gauss_beam_rot_coarse)

# ============================================
# BUILD FILENAME TAG
# ============================================
if sigma_deg is not None:
    beam_tag = f"gaussian_beam_sigma_{sigma_deg}deg"
else:
    beam_tag = f"gaussian_beam_{diameter}m"
beam_tag += f"_decay_{decay_rate}dBdeg_start_{decay_start_za}deg"

# ============================================
# SAVE TO DISK
# ============================================
save = "OFF" # Set to "OFF" to skip saving files
if save == "ON":
    save_out = "/home/herastore02-1/HERA_Validation_rchandra/"
    if rotate_gauss:
        filename = save_out + f'{beam_tag}_rttd_za_az_{theta0_deg}_{phi0_deg}.fits'
    else:
        filename = save_out + f'{beam_tag}.fits'
    gauss_beam.write_beamfits(filename, clobber=True)
    print(f"\nSaved: {filename}")

    if not rotate_gauss:
        healpix_filename = save_out + f'{beam_tag}_healpix.fits'

        gauss_beam_hpx = airy_rotated_to_healpix_efield(
            gauss_beam,
            nside=64,
            outfilename=healpix_filename,
            clobber=True,
        )

        print("Wrote HEALPix efield beam to:", healpix_filename)
        print("beam_type:", gauss_beam_hpx.beam_type)
        print("pixel_coordinate_system:", gauss_beam_hpx.pixel_coordinate_system)
        print("nside:", getattr(gauss_beam_hpx, "nside", None))

# ============================================
# CHECK BASIC METADATA
# ============================================
print("\n" + "=" * 60)
print("GAUSSIAN BEAM METADATA")
print("=" * 60)
print(f"Beam is valid:       {gauss_beam.check()}")
print(f"Telescope Name:      {gauss_beam.telescope_name}")
print(f"Feed Name:           {gauss_beam.feed_name}")
print(f"Model Name:          {gauss_beam.model_name}")
print(f"Beam type:           {gauss_beam.beam_type}")
print(f"Coordinate system:   {gauss_beam.pixel_coordinate_system}")
print(f"Data normalization:  {gauss_beam.data_normalization}")
print(f"Data shape:          {gauss_beam.data_array.shape}")
print(f"Data dtype:          {gauss_beam.data_array.dtype}")
print(f"Freq range:          {gauss_beam.freq_array.min()/1e6:.1f} - {gauss_beam.freq_array.max()/1e6:.1f} MHz")
print(f"Num frequencies:     {gauss_beam.Nfreqs}")
print(f"Naxes1 (azimuth):    {gauss_beam.Naxes1}")
print(f"Naxes2 (zenith):     {gauss_beam.Naxes2}")
print(f"Feed array:          {gauss_beam.feed_array}")
print(f"Feed angle:          {gauss_beam.feed_angle}")
print(f"NaNs in beam:        {np.isnan(gauss_beam.data_array).sum()}")
print(f"Infs in beam:        {np.isinf(gauss_beam.data_array).sum()}")
print(f"History:             {gauss_beam.history}")

# ============================================
# PLOT BEAM PATTERN
# ============================================
freq_idx = np.argmin(np.abs(freq_array - 150e6))
actual_freq = freq_array[freq_idx] / 1e6
za_deg = np.rad2deg(gauss_beam.axis2_array)

e_theta = gauss_beam.data_array[0, 0, freq_idx, :, 0]
e_phi = gauss_beam.data_array[1, 0, freq_idx, :, 0]
power = np.abs(e_theta)**2 + np.abs(e_phi)**2
power_db = 10 * np.log10(power / np.nanmax(power) + 1e-10)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(za_deg, power_db, 'b-', linewidth=2, label=f'Gaussian beam ({decay_rate} dB/deg decay)')
ax.axvline(decay_start_za, color='red', linestyle='--', linewidth=1.5,
            label=f'Decay start (ZA={decay_start_za}°)')
ax.axhline(-40, color='gray', linestyle=':', alpha=0.7, label='Floor (-40 dB)')

ax.set_xlabel('Zenith Angle (degrees)', fontsize=12)
ax.set_ylabel('Normalized Power (dB)', fontsize=12)
if sigma_deg is not None:
    ax.set_title(f'Gaussian Beam (σ={sigma_deg}°, {decay_rate} dB/deg decay from ZA={decay_start_za}°) at {actual_freq:.1f} MHz', fontsize=14)
else:
    ax.set_title(f'Gaussian Beam (D={diameter}m, {decay_rate} dB/deg decay from ZA={decay_start_za}°) at {actual_freq:.1f} MHz', fontsize=14)
ax.set_xlim(0, 180)
# ax.set_ylim(-60, 5)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

ax.fill_between([decay_start_za, 180], -60, 5, alpha=0.1, color='gray')
ax.text(decay_start_za/2, -55, 'No decay', ha='center', fontsize=10, style='italic')
ax.text((decay_start_za + 180)/2, -55, 'Exponential decay', ha='center', fontsize=10, style='italic')

plt.tight_layout()
plot_filename = f'{beam_tag}.png'
plt.savefig(plot_filename, dpi=150)
plt.show()

if rotate_gauss:
    freq_idx = np.argmin(np.abs(freq_array - 150e6))
    actual_freq = freq_array[freq_idx] / 1e6
    za_deg = np.rad2deg(fine_rot_gauss_beam.axis2_array)

    e_theta = fine_rot_gauss_beam.data_array[0, 0, freq_idx, :, 0]
    e_phi = fine_rot_gauss_beam.data_array[1, 0, freq_idx, :, 0]
    power = np.abs(e_theta)**2 + np.abs(e_phi)**2
    power_db = 10 * np.log10(power / np.nanmax(power) + 1e-10)

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(za_deg, power_db, 'b-', linewidth=2, label=f'Gaussian beam ({decay_rate} dB/deg decay)')
    ax.axvline(decay_start_za, color='red', linestyle='--', linewidth=1.5,
                label=f'Decay start (ZA={decay_start_za}°)')
    ax.axhline(-40, color='gray', linestyle=':', alpha=0.7, label='Floor (-40 dB)')

    ax.set_xlabel('Zenith Angle (degrees)', fontsize=12)
    ax.set_ylabel('Normalized Power (dB)', fontsize=12)
    ax.set_title(f'Rotated Gaussian Beam at {actual_freq:.1f} MHz', fontsize=14)
    ax.set_xlim(0, 180)
    ax.set_ylim(-60, 5)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    ax.fill_between([decay_start_za, 180], -60, 5, alpha=0.1, color='gray')
    ax.text(decay_start_za/2, -55, 'No decay', ha='center', fontsize=10, style='italic')
    ax.text((decay_start_za + 180)/2, -55, 'Exponential decay', ha='center', fontsize=10, style='italic')

    plt.tight_layout()
    plot_filename = f'{beam_tag}_rotated.png'
    # plt.savefig(plot_filename, dpi=150)
    plt.show()

print("Done!")

11. Gaussian beam math intuition

# Analytic Form of the Gaussian Beam Generator



This cell summarizes the exact functional form implemented by `create_gaussian_uvbeam` in this notebook.



## 1. Core Gaussian E-field envelope



Let $\theta$ be zenith angle in radians, $\phi$ be azimuth, and $f$ be frequency. Before any optional rotation, the generated beam is azimuth-independent, so it depends only on $(\theta, f)$.



The scalar Gaussian voltage pattern is



$$

G_0(\theta, f) = \exp\!\left[-\frac{1}{2}\left(\frac{\theta}{\sigma(f)}\right)^2\right].

$$



The width $\sigma(f)$ is defined in one of two ways.



### Fixed-sigma mode



If `sigma_deg` is supplied, then the width is frequency-independent:



$$

\sigma(f) = \sigma_0 = \frac{\pi}{180}\,\sigma_{\rm deg}.

$$



### Diameter-based mode



If `diameter = D` is supplied instead, then the notebook uses a diffraction-limited FWHM:



$$

\lambda(f) = \frac{c}{f},

$$



$$

{\rm FWHM}(f) = 1.02\,\frac{\lambda(f)}{D} = 1.02\,\frac{c}{Df},

$$



and converts to Gaussian sigma via



$$

\sigma(f) = \frac{{\rm FWHM}(f)}{2\sqrt{2\ln 2}} = \frac{1.02\,c}{2\sqrt{2\ln 2}\,D\,f}.

$$



So in this mode the beam width scales as $\sigma(f) \propto f^{-1}$.



## 2. Below-horizon exponential suppression



The notebook then multiplies the Gaussian by a piecewise attenuation in amplitude, written in dB per degree beyond a chosen zenith angle.



Let



- $\theta_{\rm deg} = 180\theta/\pi$

- $\theta_{\rm start}$ be `decay_start_za_deg`

- $r$ be `decay_rate_db_per_deg`

- $s_{\rm floor}$ be `suppression_floor_db`



The suppression in dB is



$$

s_{\rm dB}(\theta) =

\begin{cases}

0, & \theta_{\rm deg} \le \theta_{\rm start}, \\

\max\!\left[-r\,(\theta_{\rm deg}-\theta_{\rm start}),\; s_{\rm floor}\right], & \theta_{\rm deg} > \theta_{\rm start}.

\end{cases}

$$



The corresponding multiplicative amplitude envelope is



$$

E(\theta) = 10^{s_{\rm dB}(\theta)/20}.

$$



Therefore the final scalar E-field amplitude generated by the function is



$$

G(\theta, f) = \exp\!\left[-\frac{1}{2}\left(\frac{\theta}{\sigma(f)}\right)^2\right]

\times

\begin{cases}

1, & \theta_{\rm deg} \le \theta_{\rm start}, \\

10^{\max[-r(\theta_{\rm deg}-\theta_{\rm start}),\; s_{\rm floor}]/20}, & \theta_{\rm deg} > \theta_{\rm start}.

\end{cases}

$$



This is the exact analytic form of the notebook beam envelope.



## 3. Jones matrix actually stored in the UVBeam



The notebook stores this as an `efield` beam and assigns the same real scalar amplitude to all four Jones-matrix entries, with an extra factor of $1/\sqrt{2}$:



$$

J_{ab}(\phi, \theta, f) = \frac{G(\theta,f)}{\sqrt{2}}, \qquad a,b \in \{1,2\}.

$$



Equivalently,



$$

J(\phi,\theta,f) = \frac{G(\theta,f)}{\sqrt{2}}

\begin{pmatrix}

1 & 1 \\

1 & 1

\end{pmatrix}.

$$



So, before any optional rotation:



- there is no intrinsic azimuthal dependence

- there is no phase term

- there are no oscillatory rings or sidelobes

- there is no additional frequency envelope beyond the $\sigma(f)$ scaling

- the bandpass is unity at all frequencies



Since $G(0,f)=1$, the beam is peak-normalized at zenith.



## 4. Power beam implied by the notebook plotting code



The plotting cell forms power using



$$

P(\theta,f) = |E_\theta|^2 + |E_\phi|^2.

$$



Because each component carries $G(\theta,f)/\sqrt{2}$, this becomes



$$

P(\theta,f) = G(\theta,f)^2.

$$



Hence the power beam is



$$

P(\theta, f) = \exp\!\left[-\left(\frac{\theta}{\sigma(f)}\right)^2\right]

\times

\begin{cases}

1, & \theta_{\rm deg} \le \theta_{\rm start}, \\

10^{\max[-r(\theta_{\rm deg}-\theta_{\rm start}),\; s_{\rm floor}]/10}, & \theta_{\rm deg} > \theta_{\rm start}.

\end{cases}

$$



## 5. Memo-style compact form



For quick reference, the notebook Gaussian beam is:



$$

\boxed{

G(\theta,f)=\exp\!\left[-\frac{1}{2}\left(\frac{\theta}{\sigma(f)}\right)^2\right]

\times

A(\theta)

}

$$



with



$$

A(\theta)=

\begin{cases}

1, & \theta_{\rm deg}\le \theta_{\rm start}, \\

10^{\max[-r(\theta_{\rm deg}-\theta_{\rm start}),\; s_{\rm floor}]/20}, & \theta_{\rm deg}>\theta_{\rm start},

\end{cases}

$$



and



$$

\sigma(f)=

\begin{cases}

\dfrac{\pi}{180}\,\sigma_{\rm deg}, & \text{fixed-sigma mode}, \\

\dfrac{1.02\,c}{2\sqrt{2\ln 2}\,D\,f}, & \text{diameter-based mode}.

\end{cases}

$$



The stored Jones beam is



$$

\boxed{

J(\phi,\theta,f)=\dfrac{G(\theta,f)}{\sqrt{2}}

\begin{pmatrix}

1 & 1 \\

1 & 1

\end{pmatrix}

}

$$



and the plotted power beam is simply



$$

\boxed{P(\theta,f)=G(\theta,f)^2.}

$$



## 6. Note on optional rotation



If `rotate_gauss = True`, the notebook later rotates the beam pattern geometrically on the sky. That does not change the intrinsic functional form above; it only remaps the same axisymmetric beam to a new pointing direction, which then introduces azimuth dependence through coordinate transformation rather than through a new envelope term.


12. Isotropic beam generator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyuvdata import UVBeam


def create_isotropic_uvbeam(
    freq_array: np.ndarray,
    az_array: np.ndarray = None,
    za_array: np.ndarray = None,
    telescope_name: str = "HERA",
    feed_name: str = "isotropic",
) -> UVBeam:
    """
    Create an isotropic (uniform) beam pattern over the full sphere in UVBeam
    format on an az_za pixel grid.

    The E-field voltage pattern is constant everywhere:
        E_theta(za, az, f) = 1 / sqrt(2)
        E_phi  (za, az, f) = 1 / sqrt(2)
    so that the total power  |E_theta|^2 + |E_phi|^2  = 1  at every point.

    Parameters
    ----------
    freq_array : np.ndarray
        Frequencies in Hz.
    az_array : np.ndarray or None
        Azimuth grid in radians.  Default: 0–359° in 1° steps.
    za_array : np.ndarray or None
        Zenith-angle grid in radians.  Default: 0–180° in 1° steps.
    telescope_name : str
        Telescope name stored in the UVBeam.
    feed_name : str
        Feed name stored in the UVBeam.

    Returns
    -------
    beam : UVBeam
        E-field UVBeam with constant response over the full sphere.
    """
    if freq_array is None:
        raise ValueError("`freq_array` is required.")

    if az_array is None:
        az_array = np.deg2rad(np.arange(0, 360, 1).astype(float))
    if za_array is None:
        za_array = np.deg2rad(np.arange(0, 181, 1).astype(float))

    freq_array = np.asarray(freq_array, dtype=float)
    n_freq = len(freq_array)
    n_az = len(az_array)
    n_za = len(za_array)

    # Constant E-field: 1/sqrt(2) so that total power = 1 everywhere
    efield_val = 1.0 / np.sqrt(2.0)

    # data_array shape: (Naxes_vec=2, Nfeeds=2, Nfreqs, Nza, Naz)
    data_array = np.full(
        (2, 2, n_freq, n_za, n_az),
        efield_val,
        dtype=np.complex128,
    )

    # Basis vectors: theta-hat and phi-hat unit vectors
    basis_vector_array = np.zeros((2, 2, n_za, n_az))
    basis_vector_array[0, 0, :, :] = 1.0   # theta component
    basis_vector_array[1, 1, :, :] = 1.0   # phi   component

    beam = UVBeam()
    beam.pixel_coordinate_system = "az_za"
    beam.beam_type = "efield"
    beam.data_normalization = "peak"
    beam.antenna_type = "simple"

    beam.telescope_name = telescope_name
    beam.feed_name = feed_name
    beam.feed_version = "1.0"
    beam.model_name = "Isotropic (uniform full-sphere)"
    beam.model_version = "1.0"
    beam.history = "Analytic isotropic beam - constant E-field over full sphere."

    beam.Naxes_vec = 2
    beam.Ncomponents_vec = 2
    beam.Nfeeds = 2
    beam.Nfreqs = n_freq
    beam.Naxes1 = n_az
    beam.Naxes2 = n_za

    beam.axis1_array = az_array
    beam.axis2_array = za_array
    beam.freq_array = freq_array

    beam.feed_array = np.array(["x", "y"])
    beam.feed_angle = np.array([np.pi / 2, 0.0])

    beam.data_array = data_array
    beam.basis_vector_array = basis_vector_array
    beam.bandpass_array = np.ones(n_freq, dtype=float)

    beam.check()
    return beam


# ============================================
# PARAMETERS
# ============================================
freq_array = np.linspace(45e6, 250e6, 206)

# ============================================
# CREATE ISOTROPIC BEAM
# ============================================
print("Creating isotropic beam (full sphere, constant response)")

iso_beam = create_isotropic_uvbeam(
    freq_array=freq_array,
    telescope_name="HERA",
    feed_name="isotropic",
)

# ============================================
# SAVE TO DISK
# ============================================
save = "ON"  # Set to "OFF" to skip saving
if save == "ON":
    save_out = "/home/herastore02-1/HERA_Validation_rchandra/"

    # az_za beamfits
    filename_azza = save_out + "isotropic_beam_fullsphere.fits"
    iso_beam.write_beamfits(filename_azza, clobber=True)
    print(f"Saved az_za beamfits: {filename_azza}")

    # HEALPix version
    healpix_filename = save_out + "isotropic_beam_fullsphere_healpix.fits"
    iso_beam_hpx = airy_rotated_to_healpix_efield(
        iso_beam,
        nside=64,
        outfilename=healpix_filename,
        clobber=True,
    )
    print(f"Saved HEALPix beamfits: {healpix_filename}")
    print(f"  nside: {getattr(iso_beam_hpx, 'nside', None)}")

# ============================================
# METADATA
# ============================================
print("\n" + "=" * 60)
print("ISOTROPIC BEAM METADATA")
print("=" * 60)
print(f"Beam is valid:       {iso_beam.check()}")
print(f"Telescope Name:      {iso_beam.telescope_name}")
print(f"Feed Name:           {iso_beam.feed_name}")
print(f"Model Name:          {iso_beam.model_name}")
print(f"Beam type:           {iso_beam.beam_type}")
print(f"Coordinate system:   {iso_beam.pixel_coordinate_system}")
print(f"Data normalization:  {iso_beam.data_normalization}")
print(f"Data shape:          {iso_beam.data_array.shape}")
print(f"Data dtype:          {iso_beam.data_array.dtype}")
print(f"Freq range:          {iso_beam.freq_array.min()/1e6:.1f} - {iso_beam.freq_array.max()/1e6:.1f} MHz")
print(f"Num frequencies:     {iso_beam.Nfreqs}")
print(f"Naxes1 (azimuth):    {iso_beam.Naxes1}")
print(f"Naxes2 (zenith):     {iso_beam.Naxes2}")
print(f"Feed array:          {iso_beam.feed_array}")
print(f"Feed angle:          {iso_beam.feed_angle}")
print(f"NaNs in beam:        {np.isnan(iso_beam.data_array).sum()}")
print(f"Infs in beam:        {np.isinf(iso_beam.data_array).sum()}")

# ============================================
# PLOT — confirm flat response
# ============================================
freq_idx = np.argmin(np.abs(freq_array - 150e6))
actual_freq = freq_array[freq_idx] / 1e6
za_deg = np.rad2deg(iso_beam.axis2_array)

e_theta = iso_beam.data_array[0, 0, freq_idx, :, 0]
e_phi   = iso_beam.data_array[1, 0, freq_idx, :, 0]
power   = np.abs(e_theta)**2 + np.abs(e_phi)**2
power_db = 10 * np.log10(power / np.nanmax(power) + 1e-30)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Linear
axes[0].plot(za_deg, power, 'b-', linewidth=2, label="Isotropic beam")
axes[0].set_xlabel("Zenith Angle (degrees)")
axes[0].set_ylabel("Total Power (linear)")
axes[0].set_title(f"Isotropic Beam — Linear @ {actual_freq:.1f} MHz")
axes[0].set_xlim(0, 180)
axes[0].set_ylim(0, 1.5)
axes[0].axhline(1.0, color='gray', ls='--', alpha=0.5, label="Power = 1")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# dB
axes[1].plot(za_deg, power_db, 'b-', linewidth=2, label="Isotropic beam")
axes[1].set_xlabel("Zenith Angle (degrees)")
axes[1].set_ylabel("Normalized Power (dB)")
axes[1].set_title(f"Isotropic Beam — dB @ {actual_freq:.1f} MHz")
axes[1].set_xlim(0, 180)
axes[1].set_ylim(-5, 1)
axes[1].axhline(0.0, color='gray', ls='--', alpha=0.5, label="0 dB")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("isotropic_beam_fullsphere.png", dpi=150)
plt.show()

print("Done!")

13. 3D beam viewer : Isotropic

In [ ]:
import plotly.graph_objects as go
import numpy as np

# --- Use the isotropic beam ---
az = iso_beam.axis1_array          # (Naz,) radians
za = iso_beam.axis2_array          # (Nza,) radians

freq_idx = np.argmin(np.abs(freq_array - 150e6))
actual_freq = freq_array[freq_idx] / 1e6

e_theta = iso_beam.data_array[0, 0, freq_idx, :, :]  # (Nza, Naz)
e_phi   = iso_beam.data_array[1, 0, freq_idx, :, :]  # (Nza, Naz)

power = np.abs(e_theta)**2 + np.abs(e_phi)**2
max_power = np.nanmax(power)
power_db = 10 * np.log10(power / max_power + 1e-30)

min_db = -40.0
power_db_clipped = np.clip(power_db, min_db, 0.0)

# Map dB to radius: [-40, 0] -> [0, 1]
radius = (power_db_clipped - min_db) / (0.0 - min_db)

AZ, ZA = np.meshgrid(az, za)

X = radius * np.sin(ZA) * np.cos(AZ)
Y = radius * np.sin(ZA) * np.sin(AZ)
Z = radius * np.cos(ZA)

# Convert grids to degrees for hover labels
AZ_deg = np.rad2deg(AZ)
ZA_deg = np.rad2deg(ZA)

hover_text = [[
    f"Az: {AZ_deg[i, j]:.1f} deg<br>"
    f"ZA: {ZA_deg[i, j]:.1f} deg<br>"
    f"Power: {power_db_clipped[i, j]:.2f} dB"
    for j in range(AZ_deg.shape[1])]
    for i in range(AZ_deg.shape[0])
]

# --- Plotly interactive 3D surface ---
fig = go.Figure(
    data=[
        go.Surface(
            x=X, y=Y, z=Z,
            surfacecolor=power_db_clipped,
            colorscale='Viridis',
            colorbar=dict(
                title=dict(text='Power (dB)', font=dict(size=14)),
                ticksuffix=' dB',
            ),
            hoverinfo='text',
            text=hover_text,
            name='Isotropic Beam',
        )
    ]
)

fig.update_layout(
    title=dict(
        text=(f'3D Isotropic Beam Radiation Pattern @ {actual_freq:.1f} MHz<br>'
              f'<sup>Uniform E-field over full sphere | '
              f'Az: 0-360 deg, ZA: 0-180 deg</sup>'),
        font=dict(size=16),
    ),
    scene=dict(
        xaxis_title='X  (sin ZA cos Az)',
        yaxis_title='Y  (sin ZA sin Az)',
        zaxis_title='Z  (cos ZA)',
        aspectmode='data',
    ),
    width=850,
    height=850,
    margin=dict(l=10, r=10, t=80, b=10),
)

fig.show()
print("Done!")

14. Isotropic beam math intuition

# Analytic Form of the Isotropic Beam Generator

This cell summarizes the exact functional form implemented by `create_isotropic_uvbeam` in this notebook.

## 1. Core E-field pattern

Let $\theta$ be zenith angle in radians, $\phi$ be azimuth, and $f$ be frequency. The isotropic beam has **no dependence** on $\theta$, $\phi$, or $f$. The scalar voltage pattern is simply

$$
I(\theta, \phi, f) = 1.
$$

There is no angular structure, no frequency scaling, no below-horizon suppression, and no sidelobe structure of any kind.

## 2. Jones matrix stored in the UVBeam

As with the Gaussian and Airy beams, the notebook stores the isotropic beam as an `efield` beam and assigns the same constant real amplitude to all four Jones-matrix entries, with a factor of $1/\sqrt{2}$:

$$
J_{ab}(\phi, \theta, f) = \frac{1}{\sqrt{2}}, \qquad a,b \in \{1,2\}.
$$

Equivalently,

$$
\boxed{
J(\phi,\theta,f) = \frac{1}{\sqrt{2}}
\begin{pmatrix}
1 & 1 \\
1 & 1
\end{pmatrix}.
}
$$

Key properties:

- The response is identical at every point on the full sphere ($\theta \in [0, \pi]$, $\phi \in [0, 2\pi)$).
- There is no phase term — all entries are purely real.
- There is no frequency dependence — the beam is the same at every channel.
- The bandpass array is unity at all frequencies.
- The beam is peak-normalized (`data_normalization = 'peak'`), and since the pattern is constant, the peak equals the value everywhere.

## 3. Power beam

The plotting code forms total power as

$$
P(\theta, \phi, f) = |E_\theta|^2 + |E_\phi|^2.
$$

Because each component carries $1/\sqrt{2}$, this becomes

$$
\boxed{
P(\theta, \phi, f) = \left(\frac{1}{\sqrt{2}}\right)^2 + \left(\frac{1}{\sqrt{2}}\right)^2 = 1
}
$$

at every point on the sphere and at every frequency. Hence in dB:

$$
P_{\rm dB}(\theta, \phi, f) = 0 \;\text{dB} \quad \forall\; \theta, \phi, f.
$$

## 4. Comparison with the Gaussian beam

| Property | Gaussian | Isotropic |
|---|---|---|
| E-field envelope | $G(\theta,f) = e^{-\theta^2/2\sigma(f)^2}\times A(\theta)$ | $I = 1$ |
| Frequency dependence | $\sigma(f) \propto f^{-1}$ (diameter mode) or none (fixed-$\sigma$) | None |
| Below-horizon decay | Exponential suppression past $\theta_{\rm start}$ | None |
| Peak location | Zenith ($\theta = 0$) | Everywhere (uniform) |
| Jones matrix | $\frac{G(\theta,f)}{\sqrt{2}}\begin{pmatrix}1&1\\1&1\end{pmatrix}$ | $\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&1\end{pmatrix}$ |
| Power beam | $G(\theta,f)^2$ | $1$ |
| 3D radiation pattern | Frequency-dependent lobe | Perfect sphere |

## 5. Memo-style compact form

$$
\boxed{
J(\phi,\theta,f) = \frac{1}{\sqrt{2}}
\begin{pmatrix}
1 & 1 \\
1 & 1
\end{pmatrix},
\qquad
P(\theta,\phi,f) = 1.
}
$$

## 6. Use case

The isotropic beam serves as a **null test reference**: when used in place of a physical beam in the power-spectrum pipeline, any deviation of the output from the expected flat-beam result reveals artifacts introduced by the beam-handling code rather than by the beam shape itself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================
# PARAMETERS
# ============================================
decay_rate     = 0.3       # dB per degree
decay_start_za = 70.0      # zenith angle where decay starts (degrees)
diameter       = 7.0       # dish diameter (meters) — used to derive a FIXED sigma
freq_array     = np.linspace(45e6, 250e6, 206)

# Derive the fixed sigma from the diffraction limit at a reference frequency.
# This gives a beam whose angular width is frozen at the ref-freq value
# and does NOT change with frequency.
ref_freq = 80e6  # Hz  — reference frequency (change as needed)
c = 299792458.0
lam_ref = c / ref_freq
fwhm_ref = 1.02 * lam_ref / diameter
sigma_ref_deg = np.rad2deg(fwhm_ref / (2.0 * np.sqrt(2.0 * np.log(2.0))))

print(f"Frequency-independent Gaussian beam")
print(f"  Diameter used:   {diameter} m")
print(f"  Ref frequency:   {ref_freq/1e6:.1f} MHz")
print(f"  Fixed sigma:     {sigma_ref_deg:.4f} deg  (= FWHM {np.rad2deg(fwhm_ref):.4f} deg)")
print(f"  Decay rate:      {decay_rate} dB/deg past ZA = {decay_start_za} deg")

# ============================================
# CREATE FREQ-INDEPENDENT GAUSSIAN BEAM
# ============================================
gauss_beam_freqconst = create_gaussian_uvbeam(
    sigma_deg=sigma_ref_deg,        # <-- fixed sigma, no freq scaling
    diameter=None,
    freq_array=freq_array,
    telescope_name='HERA',
    feed_name='gaussian_freqconst',
    below_horizon='exponential',
    decay_rate_db_per_deg=decay_rate,
    decay_start_za_deg=decay_start_za,
)

# ============================================
# BUILD FILENAME TAG
# ============================================
beam_tag_fc = (f"gaussian_beam_{diameter}m_freqconst_ref{ref_freq/1e6:.0f}MHz"
               f"_decay_{decay_rate}dBdeg_start_{decay_start_za}deg")

# ============================================
# SAVE TO DISK
# ============================================
save = "ON"  # Set to "OFF" to skip
if save == "ON":
    save_out = "/home/herastore02-1/HERA_Validation_rchandra/"

    filename_fc = save_out + f'{beam_tag_fc}.fits'
    gauss_beam_freqconst.write_beamfits(filename_fc, clobber=True)
    print(f"\nSaved az_za beamfits: {filename_fc}")

    healpix_filename_fc = save_out + f'{beam_tag_fc}_healpix.fits'
    gauss_beam_freqconst_hpx = airy_rotated_to_healpix_efield(
        gauss_beam_freqconst,
        nside=64,
        outfilename=healpix_filename_fc,
        clobber=True,
    )
    print(f"Saved HEALPix beamfits: {healpix_filename_fc}")

# ============================================
# METADATA
# ============================================
print("\n" + "=" * 60)
print("FREQ-CONSTANT GAUSSIAN BEAM METADATA")
print("=" * 60)
print(f"Beam is valid:       {gauss_beam_freqconst.check()}")
print(f"Model Name:          {gauss_beam_freqconst.model_name}")
print(f"Data shape:          {gauss_beam_freqconst.data_array.shape}")
print(f"Freq range:          {gauss_beam_freqconst.freq_array.min()/1e6:.1f} – "
      f"{gauss_beam_freqconst.freq_array.max()/1e6:.1f} MHz")

# ============================================
# VERIFY: beam is identical across frequencies
# ============================================
slice_first = gauss_beam_freqconst.data_array[0, 0, 0, :, 0].real
slice_last  = gauss_beam_freqconst.data_array[0, 0, -1, :, 0].real
print(f"\nMax abs diff between first & last freq slice: "
      f"{np.max(np.abs(slice_first - slice_last)):.2e}  (should be 0)")

# ============================================
# PLOT: compare freq-dependent vs freq-constant
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

za_deg = np.rad2deg(gauss_beam_freqconst.axis2_array)

# Panel 1: freq-constant beam at three frequencies
for f_mhz, color, ls in [(70, 'blue', '-'), (80, 'green', '--'), (90, 'red', ':')]:
    fidx = np.argmin(np.abs(freq_array - f_mhz * 1e6))
    e_t = gauss_beam_freqconst.data_array[0, 0, fidx, :, 0]
    e_p = gauss_beam_freqconst.data_array[1, 0, fidx, :, 0]
    pwr = np.abs(e_t)**2 + np.abs(e_p)**2
    pwr_db = 10 * np.log10(pwr / np.nanmax(pwr) + 1e-30)
    axes[0].plot(za_deg, pwr_db, color=color, ls=ls, lw=2,
                 label=f'{freq_array[fidx]/1e6:.0f} MHz')

axes[0].axvline(decay_start_za, color='red', ls='--', lw=1, alpha=0.5)
axes[0].set_xlabel('Zenith Angle (degrees)')
axes[0].set_ylabel('Normalized Power (dB)')
axes[0].set_title(f'Freq-Constant Gaussian (sigma={sigma_ref_deg:.2f} deg)')
axes[0].set_xlim(0, 180)
axes[0].set_ylim(-60, 5)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: overlay freq-constant (150 MHz) vs freq-dependent (diameter mode)
# Re-create a freq-dependent beam for comparison
gauss_beam_freqdep = create_gaussian_uvbeam(
    sigma_deg=None,
    diameter=diameter,
    freq_array=freq_array,
    telescope_name='HERA',
    feed_name='gaussian_freqdep',
    below_horizon='exponential',
    decay_rate_db_per_deg=decay_rate,
    decay_start_za_deg=decay_start_za,
)

for f_mhz, color in [(70, 'blue'), (80, 'green'), (90, 'red')]:
    fidx = np.argmin(np.abs(freq_array - f_mhz * 1e6))
    # Freq-dependent
    e_t = gauss_beam_freqdep.data_array[0, 0, fidx, :, 0]
    e_p = gauss_beam_freqdep.data_array[1, 0, fidx, :, 0]
    pwr = np.abs(e_t)**2 + np.abs(e_p)**2
    pwr_db = 10 * np.log10(pwr / np.nanmax(pwr) + 1e-30)
    axes[1].plot(za_deg, pwr_db, color=color, ls='-', lw=2,
                 label=f'freq-dep {freq_array[fidx]/1e6:.0f} MHz')
    # Freq-constant
    e_t2 = gauss_beam_freqconst.data_array[0, 0, fidx, :, 0]
    e_p2 = gauss_beam_freqconst.data_array[1, 0, fidx, :, 0]
    pwr2 = np.abs(e_t2)**2 + np.abs(e_p2)**2
    pwr_db2 = 10 * np.log10(pwr2 / np.nanmax(pwr2) + 1e-30)
    axes[1].plot(za_deg, pwr_db2, color=color, ls='--', lw=2,
                 label=f'freq-const {freq_array[fidx]/1e6:.0f} MHz')

axes[1].axvline(decay_start_za, color='red', ls='--', lw=1, alpha=0.5)
axes[1].set_xlabel('Zenith Angle (degrees)')
axes[1].set_ylabel('Normalized Power (dB)')
axes[1].set_title(f'Freq-Dependent vs Freq-Constant (D={diameter}m)')
axes[1].set_xlim(0, 180)
axes[1].set_ylim(-60, 5)
axes[1].legend(fontsize=9, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{beam_tag_fc}.png', dpi=150)
plt.show()

del gauss_beam_freqdep
print("Done!")

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from pyuvdata import UVBeam

# ============================================
# COMPARE TWO BEAM FILES ON ONE 2D CUT
# ============================================
# Set these to your two beamfits files before running.
beam_file_1 = '/home/herastore02-1/HERA_Validation_rchandra/gaussian_beam_7.0m_decay_0.3dBdeg_start_70.0deg.fits'
beam_file_2 = '/home/herastore02-1/HERA_Validation_rchandra/airy_beam_7.0m_decay_0.3dBdeg_start_70.0deg.fits'

# Plot controls
plot_freq_mhz = 80.0
az_plot_deg = 0.0
feed_ind = 0
vector_inds = (0, 1)
labels = ("Beam 1", "Beam 2")
normalize_each_beam = True
plot_in_db = True
floor_db = -80.0


def load_beam(path_like: str | Path) -> UVBeam:
    beam = UVBeam()
    beam.read_beamfits(str(path_like))
    if beam.pixel_coordinate_system != "az_za":
        raise ValueError(
            f"Expected an az_za beamfits file, got {beam.pixel_coordinate_system!r} for {path_like}."
        )
    if beam.beam_type != "efield":
        raise ValueError(
            f"Expected an efield beamfits file, got {beam.beam_type!r} for {path_like}."
        )
    return beam


def beam_power_cut(
    beam: UVBeam,
    target_freq_mhz: float,
    az_deg: float,
    feed_index: int = 0,
    vec_indices: tuple[int, int] = (0, 1),
) -> tuple[np.ndarray, np.ndarray, float, float]:
    target_freq_hz = target_freq_mhz * 1e6
    freq_idx = int(np.argmin(np.abs(beam.freq_array - target_freq_hz)))
    az_idx = int(np.argmin(np.abs(np.rad2deg(beam.axis1_array) - az_deg)))

    za_deg = np.rad2deg(beam.axis2_array)
    e_field_1 = beam.data_array[vec_indices[0], feed_index, freq_idx, :, az_idx]
    e_field_2 = beam.data_array[vec_indices[1], feed_index, freq_idx, :, az_idx]
    power = np.abs(e_field_1) ** 2 + np.abs(e_field_2) ** 2

    return za_deg, power, beam.freq_array[freq_idx] / 1e6, np.rad2deg(beam.axis1_array[az_idx])


if beam_file_1 is None or beam_file_2 is None:
    print("Set beam_file_1 and beam_file_2 to two az_za efield beamfits paths, then rerun this cell.")
else:
    beam_path_1 = Path(beam_file_1).expanduser()
    beam_path_2 = Path(beam_file_2).expanduser()

    beam_1 = load_beam(beam_path_1)
    beam_2 = load_beam(beam_path_2)

    za_deg_1, power_1, actual_freq_mhz_1, actual_az_deg_1 = beam_power_cut(
        beam_1,
        target_freq_mhz=plot_freq_mhz,
        az_deg=az_plot_deg,
        feed_index=feed_ind,
        vec_indices=vector_inds,
    )
    za_deg_2, power_2, actual_freq_mhz_2, actual_az_deg_2 = beam_power_cut(
        beam_2,
        target_freq_mhz=plot_freq_mhz,
        az_deg=az_plot_deg,
        feed_index=feed_ind,
        vec_indices=vector_inds,
    )

    if normalize_each_beam:
        power_1 = power_1 / np.nanmax(power_1)
        power_2 = power_2 / np.nanmax(power_2)

    if plot_in_db:
        plot_y_1 = 10 * np.log10(np.maximum(power_1, 10 ** (floor_db / 10)))
        plot_y_2 = 10 * np.log10(np.maximum(power_2, 10 ** (floor_db / 10)))
        y_label = "Normalized Power (dB)" if normalize_each_beam else "Power (dB)"
    else:
        plot_y_1 = power_1
        plot_y_2 = power_2
        y_label = "Normalized Power" if normalize_each_beam else "Power"

    plt.figure(figsize=(12, 6))
    plt.plot(za_deg_1, plot_y_1, linewidth=2, label=f"{labels[0]} | {beam_path_1.name}")
    plt.plot(za_deg_2, plot_y_2, linewidth=2, linestyle="--", label=f"{labels[1]} | {beam_path_2.name}")
    plt.xlabel("Zenith Angle (degrees)")
    plt.ylabel(y_label)
    plt.title(
        "Beam Comparison "
        f"@ {actual_freq_mhz_1:.1f}/{actual_freq_mhz_2:.1f} MHz, "
        f"az={actual_az_deg_1:.1f}/{actual_az_deg_2:.1f} deg"
    )
    plt.xlim(0, 180)
    if plot_in_db:
        plt.ylim(floor_db, 5)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Read and plot beam

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from pyuvdata import UVBeam


# ============================================
# USER INPUTS
# ============================================
# Set these to the directory and beamfits filename you want to inspect.
# If you already have a full path, put it in read_beam_dir and set
# read_beam_filename = None.
read_beam_dir = "/home/herastore02-1/HERA_Validation_rchandra/"
read_beam_filename = "airy_beam_14.0m_freqconst_ref80MHz_decay_0.3dBdeg_start_70.0deg.fits"

# Plot/extraction controls. The ZA cut uses the azimuth closest to
# read_beam_az_cut_deg and the frequency closest to read_beam_target_freq_mhz.
read_beam_target_freq_mhz = 80.0
read_beam_az_cut_deg = 0.0
read_beam_feed_index = 0
read_beam_vec_indices = (0, 1)
read_beam_floor_db = -80.0
read_beam_plot_in_db = True
read_beam_save_png = False
read_beam_png_path = None  # if None and read_beam_save_png=True, uses filename with .png


def _angular_distance_deg(angle_deg, target_deg):
    """Smallest signed angular separation, in degrees, for circular azimuth lookup."""
    return (np.asarray(angle_deg) - target_deg + 180.0) % 360.0 - 180.0


def read_azza_efield_beam(beam_dir, beam_filename=None):
    """
    Read an az_za efield UVBeam from a beamfits file.

    Parameters
    ----------
    beam_dir : str or Path
        Directory containing the beamfits file, or a full beamfits path if
        beam_filename is None.
    beam_filename : str or None
        Beamfits filename inside beam_dir.

    Returns
    -------
    beam : UVBeam
        Loaded pyuvdata UVBeam object.
    beam_path : Path
        Full path used for the read.
    """
    beam_path = Path(beam_dir).expanduser()
    if beam_filename is not None:
        beam_path = beam_path / beam_filename

    beam = UVBeam()
    beam.read_beamfits(str(beam_path))

    if beam.pixel_coordinate_system != "az_za":
        raise ValueError(
            f"Expected an az_za beamfits file, got {beam.pixel_coordinate_system!r} for {beam_path}."
        )
    if beam.beam_type != "efield":
        raise ValueError(
            f"Expected an efield beamfits file, got {beam.beam_type!r} for {beam_path}."
        )

    return beam, beam_path


def beam_za_power_cut(
    beam,
    target_freq_mhz=150.0,
    az_deg=0.0,
    feed_index=0,
    vec_indices=(0, 1),
):
    """
    Extract normalized power versus zenith angle for one azimuth cut.

    This matches the plot_airy_za_cut convention:
        power(ZA) = |E_vec0(ZA, az)|^2 + |E_vec1(ZA, az)|^2

    Returns
    -------
    za_deg : ndarray
        Zenith-angle axis in degrees.
    norm_power : ndarray
        Linear power normalized by its maximum along this cut.
    actual_freq_mhz : float
        Frequency actually selected from the beam file.
    actual_az_deg : float
        Azimuth actually selected from the beam file.
    """
    target_freq_hz = target_freq_mhz * 1e6
    freq_idx = int(np.argmin(np.abs(beam.freq_array - target_freq_hz)))

    az_array_deg = np.rad2deg(beam.axis1_array)
    az_idx = int(np.argmin(np.abs(_angular_distance_deg(az_array_deg, az_deg))))

    za_deg = np.rad2deg(beam.axis2_array)
    e_vec0 = beam.data_array[vec_indices[0], feed_index, freq_idx, :, az_idx]
    e_vec1 = beam.data_array[vec_indices[1], feed_index, freq_idx, :, az_idx]
    power = np.abs(e_vec0) ** 2 + np.abs(e_vec1) ** 2

    peak_power = np.nanmax(power)
    if not np.isfinite(peak_power) or peak_power <= 0:
        raise ValueError("Cannot normalize beam power: peak power is non-positive or non-finite.")

    norm_power = power / peak_power
    actual_freq_mhz = beam.freq_array[freq_idx] / 1e6
    actual_az_deg = az_array_deg[az_idx]

    return za_deg, norm_power, actual_freq_mhz, actual_az_deg


def read_and_plot_beam_za_cut(
    beam_dir,
    beam_filename=None,
    target_freq_mhz=150.0,
    az_deg=0.0,
    feed_index=0,
    vec_indices=(0, 1),
    plot_in_db=True,
    floor_db=-80.0,
    save_png=False,
    png_path=None,
):
    """
    Read a beamfits file, plot one ZA cut, and return the cut arrays.

    The returned dict contains za_deg and norm_power arrays for downstream use.
    """
    beam, beam_path = read_azza_efield_beam(beam_dir, beam_filename=beam_filename)
    za_deg, norm_power, actual_freq_mhz, actual_az_deg = beam_za_power_cut(
        beam,
        target_freq_mhz=target_freq_mhz,
        az_deg=az_deg,
        feed_index=feed_index,
        vec_indices=vec_indices,
    )

    floor_linear = 10 ** (floor_db / 10.0)
    power_db = 10.0 * np.log10(np.maximum(norm_power, floor_linear))

    plot_y = power_db if plot_in_db else norm_power
    y_label = "Normalized Power (dB)" if plot_in_db else "Normalized Power"

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(za_deg, plot_y, "b-", linewidth=2, label=beam_path.name)
    ax.set_xlabel("Zenith Angle (degrees)", fontsize=12)
    ax.set_ylabel(y_label, fontsize=12)
    ax.set_title(
        f"Beam ZA cut at {actual_freq_mhz:.1f} MHz, az={actual_az_deg:.1f} deg\n"
        f"{beam_path.name}",
        fontsize=13,
    )
    ax.set_xlim(0, 180)
    if plot_in_db:
        ax.set_ylim(floor_db, 5)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()

    saved_png_path = None
    if save_png:
        saved_png_path = Path(png_path).expanduser() if png_path is not None else beam_path.with_suffix(".png")
        fig.savefig(saved_png_path, dpi=150)
        print(f"Plot saved: {saved_png_path}")

    plt.show()

    return {
        "beam": beam,
        "beam_path": beam_path,
        "za_deg": za_deg,
        "norm_power": norm_power,
        "power_db": power_db,
        "actual_freq_mhz": actual_freq_mhz,
        "actual_az_deg": actual_az_deg,
        "png_path": saved_png_path,
        "fig": fig,
        "ax": ax,
    }


read_beam_result = read_and_plot_beam_za_cut(
    read_beam_dir,
    beam_filename=read_beam_filename,
    target_freq_mhz=read_beam_target_freq_mhz,
    az_deg=read_beam_az_cut_deg,
    feed_index=read_beam_feed_index,
    vec_indices=read_beam_vec_indices,
    plot_in_db=read_beam_plot_in_db,
    floor_db=read_beam_floor_db,
    save_png=read_beam_save_png,
    png_path=read_beam_png_path,
)

# Arrays requested for downstream use.
read_beam_za_deg = read_beam_result["za_deg"]
read_beam_norm_power = read_beam_result["norm_power"]
